In [268]:
import sys
!{sys.executable} -m pip install seaborn

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import plotly.express as px

import plotly.io as pio
pd.set_option('display.max_columns', None)  # mostra todas as colunas
pd.set_option('display.max_rows', None)     # (opcional) mostra todas as linhas
pd.set_option('display.max_colwidth', None) # mostra conteúdo completo das células

In [270]:
train_data_path = "train_data/train_data.shp"
test_data_path = "test_data/test_data.shp"
test_answer_data_path = "test_answer_data/test_answer_data.shp"

In [271]:
train_data = gpd.read_file(train_data_path)
test_data = gpd.read_file(test_data_path)
test_answer_data = gpd.read_file(test_answer_data_path)

train_data = train_data.drop(columns=['soma_indic'], axis=1)
train_data = train_data.rename(columns={'cls': 'queimada'})

In [272]:
# print("Treinamento:\n", train_data.describe())
# print()
# print("Teste:\n", test_data.describe())
# print()
# print("Resposta:\n", test_answer_data.describe())

In [313]:
y = train_data.queimada

features = ["dnbr_1", "dnbrswir_1", "dndvi_1"]
answer_features = ["dnbr_1", "dnbrswir_1", "dndvi_1","queimada"]
X = train_data[features]
X_test = test_data[features]
X_answer = test_answer_data[answer_features]

In [325]:
def build_model():
    return RandomForestClassifier(
        class_weight="balanced",
        random_state=1,
        n_jobs=-1,
    )


def train_model(model, X, y):
    model.fit(X, y)
    return model


def predict(model, X):
    return model.predict(X)

In [326]:
queimada_model = build_model()

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

scoring = {"f1": "f1", "recall": "recall", "precision": "precision"}
grid_search = GridSearchCV(
    estimator=queimada_model,
    param_grid=param_grid,
    cv=5,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y)

best_model = grid_search.best_estimator_

print("Melhores parâmetros:")
print(grid_search.best_params_)

predicts = best_model.predict(X_test)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Melhores parâmetros:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [327]:
importances = best_model.feature_importances_

pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values("importance", ascending=False)

,feature,importance
1,dnbrswir_1,0.367479
0,dnbr_1,0.364039
2,dndvi_1,0.268482


In [328]:
# Valores verdadeiros
y_true = X_answer["queimada"]

# Matriz de confusão
cm = confusion_matrix(y_true, predicts)
print("Matriz de Confusão:")
print(cm)

# Métricas
print("\nEstatísticas:")
print(f"Acurácia: {accuracy_score(y_true, predicts)*100:.2f}")
print(f"Precisão: {precision_score(y_true, predicts)*100:.2f}")
print(f"Recall: {recall_score(y_true, predicts)*100:.2f}")
print(f"F1-score: {f1_score(y_true, predicts)*100:.2f}")

Matriz de Confusão:
[[8393   43]
 [  66 1498]]

Estatísticas:
Acurácia: 98.91
Precisão: 97.21
Recall: 95.78
F1-score: 96.49


In [329]:
# Teste validando com 2 dos 3 índices

In [330]:
train_data_2_path = "train_data/train_data_2_ind.shp"
test_answer_data_2_path = "test_answer_data/test_answer_data_2_ind.shp"

In [331]:
train_data_2 = gpd.read_file(train_data_2_path)
test_answer_data_2 = gpd.read_file(test_answer_data_2_path)

train_data_2 = train_data_2.drop(columns=['cls'], axis=1)

In [332]:
y2 = train_data_2.qmd_2_ind

features2 = ["dnbr_1", "dnbrswir_1", "dndvi_1"]
answer_features2 = ["dnbr_1", "dnbrswir_1", "dndvi_1","qmd_2_ind"]
X2 = train_data_2[features]
X_test2 = test_data[features]
X_answer2 = test_answer_data_2[answer_features2]

In [333]:
queimada_model_2 = RandomForestClassifier(
    class_weight="balanced",
    random_state=1,
    n_jobs=-1,
)

# queimada_model_2.fit(X2,y2)

In [334]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

scoring = {"f1": "f1", "recall": "recall", "precision": "precision"}
grid_search = GridSearchCV(
    estimator=queimada_model,
    param_grid=param_grid,
    cv=5,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y)

best_model_2 = grid_search.best_estimator_

print("Melhores parâmetros:")
print(grid_search.best_params_)

predicts2 = best_model_2.predict(X_test)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Melhores parâmetros:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [335]:
importances = best_model_2.feature_importances_

pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values("importance", ascending=False)

,feature,importance
1,dnbrswir_1,0.367479
0,dnbr_1,0.364039
2,dndvi_1,0.268482


In [336]:
# Valores verdadeiros
y_true = X_answer2["qmd_2_ind"]

# Matriz de confusão
cm = confusion_matrix(y_true, predicts2)
print("Matriz de Confusão:")
print(cm)

# Métricas
print("\nEstatísticas:")
print(f"Acurácia: {accuracy_score(y_true, predicts2)*100:.2f}")
print(f"Precisão: {precision_score(y_true, predicts2)*100:.2f}")
print(f"Recall: {recall_score(y_true, predicts2)*100:.2f}")
print(f"F1-score: {f1_score(y_true, predicts2)*100:.2f}")


# Com os 3 índices:
# Matriz de Confusão:
# [[8378   58]
#  [  74 1490]]
# Acurácia: 98.84 Precisão: 96.89
# Recall: 95.65 F1-score: 96.27

# Com grid search:
# Matriz de Confusão:
# [[8393   43]
#  [  66 1498]]
# Acurácia: 98.91 Precisão: 97.21
# Recall: 95.78 F1-score: 96.49



# Com pelo menos 2 dos 3 índices:
# Matriz de Confusão:
# [[7817   52]
#  [  81 2050]]
# Acurácia: 98.67 Precisão: 97.53
# Recall: 96.20 F1-score: 96.86

# Com grid search:
# Matriz de Confusão:
# [[7869    0]
#  [ 590 1541]]
# Acurácia: 94.10 Precisão: 100.00
# Recall: 72.31 F1-score: 83.93

Matriz de Confusão:
[[7869    0]
 [ 590 1541]]

Estatísticas:
Acurácia: 94.10
Precisão: 100.00
Recall: 72.31
F1-score: 83.93
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   0.4s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=10, n_estimators=300; total time=   1.2s
[CV] END max_depth=None, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   0.5s
[CV] END max_depth=None, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   1.1s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=   0.4s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time=   1.2s
[CV] END max_depth=10, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   1.2s
[CV] END max_depth=20, min_samples_leaf=1, min_samples_split=10, n_estimators=300; total time=   1.2s
[CV] END max_depth=20, min_samples_leaf=4, min_samples_s

In [ ]:
# Com os 3 índices:
# Matriz de Confusão:
# [[8378   58]
#  [  74 1490]]
# Acurácia: 98.84 Precisão: 96.89
# Recall: 95.65 F1-score: 96.27

# Com grid search:
# Matriz de Confusão:
# [[8393   43]
#  [  66 1498]]
# Acurácia: 98.91 Precisão: 97.21
# Recall: 95.78 F1-score: 96.49



# Com pelo menos 2 dos 3 índices:
# Matriz de Confusão:
# [[7817   52]
#  [  81 2050]]
# Acurácia: 98.67 Precisão: 97.53
# Recall: 96.20 F1-score: 96.86

# Com grid search:
# Matriz de Confusão:
# [[7869    0]
#  [ 590 1541]]
# Acurácia: 94.10 Precisão: 100.00
# Recall: 72.31 F1-score: 83.93